# LeetCode 1287 — Element Appearing More Than 25% In Sorted Array bằng Python

**Link đề:** https://leetcode.com/problems/element-appearing-more-than-25-in-sorted-array/
**Độ khó:** Easy · **Chủ đề:** Array

```python
class Solution:
    def findSpecialInteger(self, arr: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Đếm tần suất bằng `dict` / `collections.Counter` — cách tổng quát, không cần mảng đã sắp xếp.
2. Khai thác **mảng đã sắp xếp**: các giá trị bằng nhau nằm **liền nhau** (thành "khối"), đếm độ dài khối với `O(1)` bộ nhớ.
3. Mẹo đẹp: phần tử chiếm > 25% thì `arr[i] == arr[i + n // 4]` với một `i` nào đó.
4. Tối ưu xuống `O(log n)` bằng **tìm kiếm nhị phân** (`bisect`) tại 3 vị trí ứng viên `n/4`, `n/2`, `3n/4`.
5. Chứng minh từng mẹo, trace, test đối chiếu, lỗi hay gặp (nhầm `>=` với `>`), biến thể và bài tập.

## 1. Đề bài và ví dụ

Cho mảng số nguyên `arr` đã **sắp xếp không giảm**. Có **đúng một** số nguyên xuất hiện **nhiều hơn 25%** số phần tử. Trả về số đó.

| `arr` | `n` | Output | Giải thích |
|---|---:|---:|---|
| `[1,2,2,6,6,6,6,7,10]` | 9 | `6` | 6 xuất hiện 4 lần, `4 > 9/4 = 2.25` |
| `[1,1]` | 2 | `1` | 1 xuất hiện 2 lần, `2 > 0.5` |

**Constraints:** `1 <= arr.length <= 10^4`, `0 <= arr[i] <= 10^5`.

"Nhiều hơn 25%" nghĩa là `count > n / 4`, tương đương `4 * count > n` (viết kiểu số nguyên để tránh số thực).

In [1]:
arr = [1, 2, 2, 6, 6, 6, 6, 7, 10]
n = len(arr)
for v in sorted(set(arr)):
    c = arr.count(v)
    print(f"{v:>2}: {c} lần | 4*count={4 * c:>2} > n={n}? {4 * c > n}")

 1: 1 lần | 4*count= 4 > n=9? False
 2: 2 lần | 4*count= 8 > n=9? False
 6: 4 lần | 4*count=16 > n=9? True
 7: 1 lần | 4*count= 4 > n=9? False
10: 1 lần | 4*count= 4 > n=9? False


## 2. Nền tảng Python cần biết

### 2.1. `dict` để đếm

`count[x] = count.get(x, 0) + 1` — `get(x, 0)` trả giá trị hiện có hoặc `0` nếu `x` chưa có key.

In [2]:
count = {}
for x in [1, 2, 2, 6, 6]:
    count[x] = count.get(x, 0) + 1
print(count)            # {1: 1, 2: 2, 6: 2}
print(count.get(9, 0))  # 0

{1: 1, 2: 2, 6: 2}
0


### 2.2. `collections.Counter` và `most_common`

`Counter(arr)` là `dict` chuyên đếm. `most_common(1)` trả list chứa **một** cặp `(giá_trị, số_lần)` lớn nhất.

In [3]:
from collections import Counter
c = Counter([1, 2, 2, 6, 6, 6, 6, 7, 10])
print(c)
print(c.most_common(1))          # [(6, 4)]
print(c.most_common(1)[0][0])    # 6

Counter({6: 4, 2: 2, 1: 1, 7: 1, 10: 1})
[(6, 4)]
6


### 2.3. Mảng đã sắp xếp: giá trị bằng nhau nằm thành **khối liền nhau**

```text
index: 0 1 2 3 4 5 6 7 8
arr  : 1 2 2 6 6 6 6 7 10
         └─┘ └─────┘
```

Khối của `6` là đoạn `[3, 6]`, độ dài `4`. Muốn biết một giá trị xuất hiện bao nhiêu lần, chỉ cần biết **vị trí đầu** và **vị trí cuối** của khối.

### 2.4. `n // 4` — chia lấy phần nguyên

`9 // 4 = 2`, `2 // 4 = 0`, `8 // 4 = 2`. Nhớ rằng `count > n / 4` ⇔ `count >= n // 4 + 1` (với số nguyên).

In [4]:
for n in [1, 2, 4, 8, 9, 10]:
    q = n // 4
    print(f"n={n:>2}: n/4={n / 4:<5} n//4={q} -> cần count >= {q + 1}")

n= 1: n/4=0.25  n//4=0 -> cần count >= 1
n= 2: n/4=0.5   n//4=0 -> cần count >= 1
n= 4: n/4=1.0   n//4=1 -> cần count >= 2
n= 8: n/4=2.0   n//4=2 -> cần count >= 3
n= 9: n/4=2.25  n//4=2 -> cần count >= 3
n=10: n/4=2.5   n//4=2 -> cần count >= 3


### 2.5. `bisect_left` và `bisect_right` — tìm biên của khối trong `O(log n)`

Với list đã sắp xếp `a` và giá trị `x`:
- `bisect_left(a, x)` = vị trí **đầu tiên** có `a[i] >= x` (chỗ chèn `x` vào bên trái các bản sao).
- `bisect_right(a, x)` = vị trí **đầu tiên** có `a[i] > x` (chỗ chèn bên phải).
- Số lần `x` xuất hiện = `bisect_right - bisect_left`.

In [5]:
from bisect import bisect_left, bisect_right
a = [1, 2, 2, 6, 6, 6, 6, 7, 10]
print(bisect_left(a, 6), bisect_right(a, 6))           # 3 7
print("số lần 6:", bisect_right(a, 6) - bisect_left(a, 6))   # 4
print(bisect_left(a, 5), bisect_right(a, 5))           # 3 3 -> 5 không có, đếm = 0

3 7
số lần 6: 4
3 3


## 3. Cách 1 — Đếm bằng `Counter` (không dùng tính sắp xếp)

Đếm mọi giá trị, trả về giá trị có `4 * count > n`. Đơn giản, đúng cho cả mảng **không** sắp xếp.

In [6]:
from collections import Counter

def special_counter(arr: list[int]) -> int:
    n = len(arr)
    for v, c in Counter(arr).items():
        if 4 * c > n:
            return v
    return -1          # không xảy ra với input hợp lệ

print(special_counter([1, 2, 2, 6, 6, 6, 6, 7, 10]))   # 6
print(special_counter([1, 1]))                         # 1

6
1


**Độ phức tạp:** thời gian `O(n)`, bộ nhớ `O(k)` với `k` = số giá trị khác nhau (tối đa `O(n)`).

## 4. Nút thắt

Cách 1 **không dùng** dữ kiện "đã sắp xếp" — dấu hiệu là còn tối ưu được:
1. Bộ nhớ: khối liền nhau ⇒ chỉ cần đếm độ dài khối hiện tại, không cần `dict`.
2. Thời gian: khối dài hơn `n/4` là khối **rất lớn** — có lẽ không cần nhìn mọi phần tử?

## 5. Cách 2 — Quét một lượt, đếm độ dài khối (run-length)

Duyệt từ trái sang phải; nếu `arr[i] == arr[i-1]` thì khối tiếp tục (`run += 1`), ngược lại khối mới (`run = 1`). Khi `4 * run > n` → trả về ngay.

In [7]:
def special_run(arr: list[int]) -> int:
    n = len(arr)
    run = 0
    for i in range(n):
        if i > 0 and arr[i] == arr[i - 1]:
            run += 1
        else:
            run = 1
        if 4 * run > n:
            return arr[i]
    return -1

print(special_run([1, 2, 2, 6, 6, 6, 6, 7, 10]))   # 6
print(special_run([1, 1]))                         # 1
print(special_run([5]))                            # 5

6
1
5


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

## 6. Insight — nhảy cách `n // 4` bước

Gọi `q = n // 4`. Nếu giá trị `v` chiếm `count >= q + 1` vị trí **liền nhau** bắt đầu từ `i`, thì phần tử ở `i + q` vẫn nằm trong khối:

```text
i, i+1, ..., i+q   <- q+1 vị trí, đều là v
```

Nên **`arr[i] == arr[i + q]`**. Ngược lại, nếu `arr[i] == arr[i + q]` thì (vì mảng sắp xếp) mọi phần tử giữa cũng bằng nhau → giá trị đó xuất hiện ≥ `q + 1 > n/4` lần → chính là đáp án.

Ví dụ `n = 9`, `q = 2`: tìm `i` với `arr[i] == arr[i + 2]`.

In [8]:
arr = [1, 2, 2, 6, 6, 6, 6, 7, 10]
q = len(arr) // 4
for i in range(len(arr) - q):
    print(f"i={i}: arr[{i}]={arr[i]:>2}, arr[{i + q}]={arr[i + q]:>2} -> {'BẰNG' if arr[i] == arr[i + q] else ''}")

i=0: arr[0]= 1, arr[2]= 2 -> 
i=1: arr[1]= 2, arr[3]= 6 -> 
i=2: arr[2]= 2, arr[4]= 6 -> 
i=3: arr[3]= 6, arr[5]= 6 -> BẰNG
i=4: arr[4]= 6, arr[6]= 6 -> BẰNG
i=5: arr[5]= 6, arr[7]= 7 -> 
i=6: arr[6]= 6, arr[8]=10 -> 


In [9]:
def special_jump(arr: list[int]) -> int:
    q = len(arr) // 4
    for i in range(len(arr) - q):
        if arr[i] == arr[i + q]:
            return arr[i]
    return -1

print(special_jump([1, 2, 2, 6, 6, 6, 6, 7, 10]))   # 6
print(special_jump([1, 1]))                         # q = 0 -> arr[0] == arr[0] -> 1

6
1


Trường hợp `n < 4`: `q = 0`, `arr[0] == arr[0]` luôn đúng → trả `arr[0]`. Có đúng không? Với `n <= 3`, một phần tử chỉ cần xuất hiện ≥ 1 lần là đã > 25%… nhưng đề nói có **đúng một** phần tử như vậy, nên mảng phải toàn một giá trị (ví dụ `[1,1]`, `[5]`, `[3,3,3]`). Vậy trả `arr[0]` là đúng.

## 7. Cách 4 — Nhị phân tại 3 ứng viên: `O(log n)`

Insight mạnh hơn: khối dài **hơn** `n/4` thì chắc chắn **phủ** ít nhất một trong 3 vị trí `n//4`, `n//2`, `3n//4`.

Lý do: 3 vị trí này chia mảng thành 4 "khe" (trước, giữa, giữa, sau), mỗi khe có **≤ n/4** phần tử nằm hẳn bên trong. Một khối dài > `n/4` không thể nằm gọn trong một khe → phải "chạm" một vị trí mốc.

Vậy chỉ có **3 ứng viên** `arr[n//4]`, `arr[n//2]`, `arr[3n//4]`. Với mỗi ứng viên, đếm số lần bằng `bisect` (mỗi lần `O(log n)`).

In [10]:
from bisect import bisect_left, bisect_right

def special_binary(arr: list[int]) -> int:
    n = len(arr)
    for pos in (n // 4, n // 2, 3 * n // 4):
        v = arr[pos]
        cnt = bisect_right(arr, v) - bisect_left(arr, v)
        if 4 * cnt > n:
            return v
    return -1

print(special_binary([1, 2, 2, 6, 6, 6, 6, 7, 10]))   # 6
print(special_binary([1, 1]))                         # 1

6
1


### Kiểm chứng "khe ≤ n/4" bằng code

Với mỗi `n`, ta đếm số phần tử nằm **hẳn** giữa các mốc và kiểm tra `4 * khe <= n`.

In [11]:
for n in range(1, 2001):
    marks = sorted({n // 4, n // 2, 3 * n // 4})
    gaps = [marks[0]]                                   # trước mốc đầu
    gaps += [marks[k + 1] - marks[k] - 1 for k in range(len(marks) - 1)]
    gaps.append(n - 1 - marks[-1])                      # sau mốc cuối
    assert all(4 * g <= n for g in gaps), (n, marks, gaps)
print("Mọi n trong 1..2000: mỗi khe có <= n/4 phần tử")
print("ví dụ n=9:", sorted({9 // 4, 9 // 2, 27 // 4}))

Mọi n trong 1..2000: mỗi khe có <= n/4 phần tử
ví dụ n=9: [2, 4, 6]


### Tự viết tìm kiếm nhị phân (không dùng `bisect`)

Để hiểu `bisect_left` hoạt động thế nào: giữ khoảng `[lo, hi)`; nếu `a[mid] < x` thì đáp án ở bên phải, ngược lại ở bên trái (kể cả `mid`).

In [12]:
def my_bisect_left(a, x):
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo

def my_bisect_right(a, x):
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] <= x:          # khác duy nhất: <=
            lo = mid + 1
        else:
            hi = mid
    return lo

a = [1, 2, 2, 6, 6, 6, 6, 7, 10]
print(my_bisect_left(a, 6), my_bisect_right(a, 6))    # 3 7
assert all(my_bisect_left(a, x) == bisect_left(a, x) and my_bisect_right(a, x) == bisect_right(a, x) for x in range(-1, 12))
print("khớp bisect chuẩn")

3 7
khớp bisect chuẩn


## 8. Trace từng bước

### Trace Cách 2 (quét khối) trên `[1,2,2,6,6,6,6,7,10]`, `n = 9`

| i | arr[i] | bằng arr[i-1]? | run | 4·run > 9? |
|---:|---:|---|---:|---|
| 0 | 1 | — | 1 | 4 > 9 ✗ |
| 1 | 2 | ✗ | 1 | ✗ |
| 2 | 2 | ✔ | 2 | 8 > 9 ✗ |
| 3 | 6 | ✗ | 1 | ✗ |
| 4 | 6 | ✔ | 2 | ✗ |
| 5 | 6 | ✔ | 3 | 12 > 9 ✔ → trả 6 |

In [13]:
def trace_run(arr):
    n = len(arr)
    run = 0
    for i in range(n):
        same = i > 0 and arr[i] == arr[i - 1]
        run = run + 1 if same else 1
        print(f"i={i}: arr[i]={arr[i]:>2}, cùng khối={same!s:<5}, run={run}, 4*run={4 * run:>2} > {n}? {4 * run > n}")
        if 4 * run > n:
            print("-> trả", arr[i])
            return arr[i]

trace_run([1, 2, 2, 6, 6, 6, 6, 7, 10])

i=0: arr[i]= 1, cùng khối=False, run=1, 4*run= 4 > 9? False
i=1: arr[i]= 2, cùng khối=False, run=1, 4*run= 4 > 9? False
i=2: arr[i]= 2, cùng khối=True , run=2, 4*run= 8 > 9? False
i=3: arr[i]= 6, cùng khối=False, run=1, 4*run= 4 > 9? False
i=4: arr[i]= 6, cùng khối=True , run=2, 4*run= 8 > 9? False
i=5: arr[i]= 6, cùng khối=True , run=3, 4*run=12 > 9? True
-> trả 6


6

`{same!s:<5}`: `!s` ép về chuỗi bằng `str(...)` rồi căn trái độ rộng 5 — để cột in thẳng hàng.

### Trace Cách 4 (nhị phân)

In [14]:
def trace_binary(arr):
    n = len(arr)
    for pos in (n // 4, n // 2, 3 * n // 4):
        v = arr[pos]
        L, R = bisect_left(arr, v), bisect_right(arr, v)
        print(f"ứng viên arr[{pos}]={v}: khối [{L}, {R}) dài {R - L}, 4*{R - L}={4 * (R - L)} > {n}? {4 * (R - L) > n}")
        if 4 * (R - L) > n:
            return v

print("->", trace_binary([1, 2, 2, 6, 6, 6, 6, 7, 10]))
print("->", trace_binary([1, 1, 1, 2, 3, 4, 5, 6, 7, 8]))

ứng viên arr[2]=2: khối [1, 3) dài 2, 4*2=8 > 9? False
ứng viên arr[4]=6: khối [3, 7) dài 4, 4*4=16 > 9? True
-> 6
ứng viên arr[2]=1: khối [0, 3) dài 3, 4*3=12 > 10? True
-> 1


## 9. Code nộp LeetCode

Bản `O(n)` / `O(1)` dễ nhớ (mẹo nhảy `n // 4`):

In [15]:
class Solution:
    def findSpecialInteger(self, arr: list[int]) -> int:
        q = len(arr) // 4
        for i in range(len(arr) - q):
            if arr[i] == arr[i + q]:
                return arr[i]
        return -1

print(Solution().findSpecialInteger([1, 2, 2, 6, 6, 6, 6, 7, 10]))   # 6

6


Bản `O(log n)`:

```python
from bisect import bisect_left, bisect_right

class Solution:
    def findSpecialInteger(self, arr: list[int]) -> int:
        n = len(arr)
        for pos in (n // 4, n // 2, 3 * n // 4):
            v = arr[pos]
            if 4 * (bisect_right(arr, v) - bisect_left(arr, v)) > n:
                return v
        return -1
```

## 10. Vì sao đúng?

- **Cách 2:** invariant — `run` luôn bằng độ dài khối chứa `arr[i]` tính đến `i` (vì mảng sắp xếp, các bản sao liền nhau). Khối của đáp án đạt độ dài > `n/4` tại một thời điểm, và không khối nào khác đạt được (đáp án duy nhất).
- **Cách 3:** đã chứng minh hai chiều ở mục 6: tồn tại `i` với `arr[i] == arr[i+q]` ⇔ có giá trị xuất hiện ≥ `q+1` lần ⇔ > `n/4` lần.
- **Cách 4:** khối của đáp án phủ một mốc (mục 7) ⇒ đáp án nằm trong 3 ứng viên; ta kiểm tra **đúng** số lần của từng ứng viên nên không trả nhầm.

## 11. Độ phức tạp và so sánh

| Cách | Thời gian | Bộ nhớ phụ | Dùng tính sắp xếp? |
|---|---|---|---|
| 1. `Counter` | `O(n)` | `O(n)` | Không |
| 2. Quét khối | `O(n)` | `O(1)` | Có |
| 3. Nhảy `n // 4` | `O(n)` | `O(1)` | Có |
| 4. Nhị phân 3 ứng viên | `O(log n)` | `O(1)` | Có |

## 12. Bộ test

Sinh input hợp lệ ngẫu nhiên: tạo mảng sắp xếp ngẫu nhiên rồi **chỉ giữ** những mảng có đúng một giá trị chiếm > 25%.

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.findSpecialInteger([1, 2, 2, 6, 6, 6, 6, 7, 10]), 6)
check("đề 2", sol.findSpecialInteger([1, 1]), 1)
check("1 phần tử", sol.findSpecialInteger([7]), 7)
check("ở đầu mảng", sol.findSpecialInteger([1, 1, 1, 2, 3, 4, 5, 6, 7, 8]), 1)
check("ở cuối mảng", sol.findSpecialInteger([1, 2, 3, 4, 5, 6, 7, 9, 9, 9]), 9)
check("toàn một giá trị", sol.findSpecialInteger([4, 4, 4, 4]), 4)
check("vừa đủ > 25% (n=8, 3 lần)", sol.findSpecialInteger([1, 2, 3, 3, 3, 4, 5, 6]), 3)

PASS đề 1: 6
PASS đề 2: 1
PASS 1 phần tử: 7
PASS ở đầu mảng: 1
PASS ở cuối mảng: 9
PASS toàn một giá trị: 4
PASS vừa đủ > 25% (n=8, 3 lần): 3


In [17]:
import random
random.seed(1287)
methods = [special_counter, special_run, special_jump, special_binary]
tested = 0
while tested < 2000:
    n = random.randint(1, 40)
    arr = sorted(random.choice([random.randint(0, 12) for _ in range(4)] + [random.randint(0, 12)]) for _ in range(n))
    winners = [v for v, c in Counter(arr).items() if 4 * c > n]
    if len(winners) != 1:
        continue
    for f in methods:
        assert f(arr) == winners[0], (f.__name__, arr)
    assert Solution().findSpecialInteger(arr) == winners[0]
    tested += 1
print("2000 mảng hợp lệ ngẫu nhiên: 4 cách + Solution đều khớp")

2000 mảng hợp lệ ngẫu nhiên: 4 cách + Solution đều khớp


## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng `>=` 25% thay vì `>` 25%

Với `n = 8`, phần tử xuất hiện **đúng 2 lần** là đúng 25%, **không** đủ điều kiện. Dùng `4 * c >= n` sẽ trả nhầm phần tử xuất hiện trước.

In [18]:
def wrong_ge(arr):
    n = len(arr)
    for v, c in Counter(arr).items():
        if 4 * c >= n:                 # SAI
            return v

arr = [1, 1, 2, 3, 3, 3, 4, 5]          # 1 chiếm đúng 25%, 3 chiếm 37.5%
print("sai :", wrong_ge(arr), "| đúng:", Solution().findSpecialInteger(arr))

sai : 1 | đúng: 3


### Lỗi 2: So sánh với `n / 4` rồi ép kiểu `int`

`count > int(n / 4)` thì **đúng**, nhưng `count >= int(n / 4)` hoặc `count > round(n / 4)` thì sai ở một số `n`. Cách an toàn: so sánh số nguyên `4 * count > n`.

In [19]:
for n in [5, 6, 7, 9, 10]:
    print(f"n={n}: cần count > {n / 4}; round(n/4)={round(n / 4)} -> dùng 'count > round' sẽ đòi count >= {round(n / 4) + 1}, đúng ra chỉ cần >= {n // 4 + 1}")

n=5: cần count > 1.25; round(n/4)=1 -> dùng 'count > round' sẽ đòi count >= 2, đúng ra chỉ cần >= 2
n=6: cần count > 1.5; round(n/4)=2 -> dùng 'count > round' sẽ đòi count >= 3, đúng ra chỉ cần >= 2
n=7: cần count > 1.75; round(n/4)=2 -> dùng 'count > round' sẽ đòi count >= 3, đúng ra chỉ cần >= 2
n=9: cần count > 2.25; round(n/4)=2 -> dùng 'count > round' sẽ đòi count >= 3, đúng ra chỉ cần >= 3
n=10: cần count > 2.5; round(n/4)=2 -> dùng 'count > round' sẽ đòi count >= 3, đúng ra chỉ cần >= 3


### Lỗi 3: Chỉ kiểm tra phần tử ở giữa `arr[n // 2]`

Khối 25%+ có thể nằm ở **đầu** hoặc **cuối** mảng, không chạm giữa.

In [20]:
def wrong_middle(arr):
    return arr[len(arr) // 2]

arr = [1, 1, 1, 2, 3, 4, 5, 6, 7, 8]
print("sai :", wrong_middle(arr), "| đúng:", Solution().findSpecialInteger(arr))

sai : 4 | đúng: 1


### Lỗi 4: Vượt chỉ số khi nhảy `i + q`

Duyệt `for i in range(len(arr))` rồi đọc `arr[i + q]`. Với input hợp lệ, đáp án luôn được tìm thấy **trước** khi vượt biên nên lỗi bị **che giấu** — test của đề vẫn qua! Nhưng chỉ cần một input không có đáp án (ví dụ khi bạn tái sử dụng hàm cho bài khác) là `IndexError`. Giới hạn đúng: `range(len(arr) - q)`.

In [21]:
def wrong_range(arr):
    q = len(arr) // 4
    for i in range(len(arr)):            # SAI: thừa q vòng cuối
        if arr[i] == arr[i + q]:
            return arr[i]

print("input hợp lệ vẫn qua:", wrong_range([1, 2, 3, 4, 5, 6, 7, 9, 9, 9]))
try:
    print(wrong_range([1, 2, 3, 4, 5, 6, 7, 8]))     # không có phần tử > 25%
except IndexError as e:
    print("IndexError:", e)

input hợp lệ vẫn qua: 9
IndexError: list index out of range


### Lỗi 5: Dùng `arr.count(v)` trong vòng lặp

`arr.count` duyệt cả mảng `O(n)`; gọi cho mỗi phần tử → `O(n²)`. Với `n = 10^4` vẫn có thể qua, nhưng là thói quen xấu.

## 14. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu:**
- "Mảng đã sắp xếp" + "đếm số lần xuất hiện" → khối liền nhau, `bisect_left/right`.
- "Xuất hiện nhiều hơn 1/k" → so sánh `arr[i]` với `arr[i + n//k]`; hoặc kiểm tra các mốc `n/k, 2n/k, …`.
- Chưa sắp xếp + "nhiều hơn n/2" → thuật toán **Boyer–Moore majority vote**; "nhiều hơn n/3" → Boyer–Moore mở rộng (LeetCode 229).

**Bài liên quan:**
- LeetCode 169 *Majority Element* (> n/2).
- LeetCode 229 *Majority Element II* (> n/3).
- LeetCode 34 *Find First and Last Position of Element in Sorted Array* (chính là `bisect_left/right`).
- LeetCode 1150 *Check If a Number Is Majority Element in a Sorted Array* (Premium).

In [22]:
# Mini: Boyer–Moore cho "nhiều hơn n/2" (mảng không cần sắp xếp)
def majority(nums):
    cand, cnt = None, 0
    for x in nums:
        if cnt == 0:
            cand = x
        cnt += 1 if x == cand else -1
    return cand

print(majority([2, 2, 1, 1, 1, 2, 2]))   # 2

2


## 15. Bài tập tự luyện

> Gợi ý ở mục 16, đáp án ở mục 17. Không gọi hàm TODO khi chạy notebook.

### Bài 1 — Tính tay
Với `arr = [0,1,2,3,3,3,3,4,5,6,7,8]` (`n = 12`): `q = n // 4` là bao nhiêu, và `i` **đầu tiên** có `arr[i] == arr[i+q]` là bao nhiêu? Gán tuple `(q, i)` vào `ex1_answer`.

In [23]:
ex1_answer = None   # TODO: (q, i)

# assert ex1_answer == (len(arr_ex1) // 4, next(i for i in range(9) if arr_ex1[i] == arr_ex1[i + 3]))
# với arr_ex1 = [0, 1, 2, 3, 3, 3, 3, 4, 5, 6, 7, 8]

### Bài 2 — Đếm số lần xuất hiện trong mảng đã sắp xếp
Viết `count_sorted(arr, x)` trong `O(log n)`.

In [24]:
def count_sorted(arr: list[int], x: int) -> int:
    # TODO
    pass

# assert count_sorted([1, 2, 2, 6, 6, 6, 6, 7, 10], 6) == 4
# assert count_sorted([1, 2, 2, 6, 6, 6, 6, 7, 10], 5) == 0

### Bài 3 — Tổng quát "> 1/k"
Viết `more_than_1_over_k(arr, k)` trả về **list tăng dần** các giá trị xuất hiện nhiều hơn `n / k` lần trong mảng đã sắp xếp, dùng `O(1)` bộ nhớ phụ (ngoài kết quả).

In [25]:
def more_than_1_over_k(arr: list[int], k: int) -> list[int]:
    # TODO: quét khối
    pass

# assert more_than_1_over_k([1, 1, 2, 2, 2, 3], 3) == [2]
# assert more_than_1_over_k([1, 1, 1, 2, 2, 2, 3], 3) == [1, 2]

### Bài 4 — Vị trí khối của đáp án
Viết `special_block(arr)` trả về `(start, end)` (bao gồm hai đầu) của khối đáp án, trong `O(log n)`.

In [26]:
def special_block(arr: list[int]) -> tuple[int, int]:
    # TODO
    pass

# assert special_block([1, 2, 2, 6, 6, 6, 6, 7, 10]) == (3, 6)

### Bài 5 — Mini challenge: Majority Element II (LeetCode 229) cho mảng đã sắp xếp
Mảng **đã sắp xếp**, tìm mọi giá trị xuất hiện > `n/3` lần trong `O(log n)` bằng cách kiểm tra các mốc.

In [27]:
def majority_third_sorted(arr: list[int]) -> list[int]:
    # TODO: mốc n//3 và 2n//3 ... hãy tự kiểm tra lại xem đã đủ chưa
    pass

# assert majority_third_sorted([1, 1, 1, 2, 3, 3, 3]) == [1, 3]
# assert majority_third_sorted([1, 2, 3]) == []

## 16. Gợi ý 3 mức

**Bài 1:** (1) `12 // 4`. (2) Khối của 3 là `[3, 6]` (4 phần tử, `4·4 = 16 > 12`). (3) Thử `i = 0, 1, 2, …` so `arr[i]` với `arr[i+3]`.

**Bài 2:** (1) Mục 2.5. (2) Hiệu hai vị trí chèn. (3) `bisect_right(arr, x) - bisect_left(arr, x)`.

**Bài 3:** (1) Như Cách 2 nhưng không trả về sớm. (2) Khi khối kết thúc (hoặc hết mảng), kiểm `k * run > n`. (3) Thêm vào kết quả rồi đặt lại `run`.

**Bài 4:** (1) Dùng Cách 4 tìm giá trị. (2) `bisect_left` cho `start`. (3) `bisect_right - 1` cho `end`.

**Bài 5:** (1) Khối > n/3 có thể không chạm cả `n//3` lẫn `2n//3` khi `n` nhỏ? Hãy kiểm "khe". (2) Thêm các mốc an toàn: `0`, `n//3`, `2n//3`, `n-1` đều là ứng viên hợp lệ. (3) Với mỗi ứng viên đếm bằng `bisect`, gom vào `set` rồi `sorted`.

## 17. Đáp án tham khảo

In [28]:
ans_ex1 = (3, 3)      # q = 12 // 4 = 3; arr[3] = 3 = arr[6]
arr_ex1 = [0, 1, 2, 3, 3, 3, 3, 4, 5, 6, 7, 8]


def ans_count_sorted(arr, x):
    return bisect_right(arr, x) - bisect_left(arr, x)


def ans_more_than_1_over_k(arr, k):
    n, res, run = len(arr), [], 0
    for i in range(n):
        run = run + 1 if i > 0 and arr[i] == arr[i - 1] else 1
        block_ends = i == n - 1 or arr[i + 1] != arr[i]
        if block_ends and k * run > n:
            res.append(arr[i])
    return res


def ans_special_block(arr):
    v = special_binary(arr)
    return bisect_left(arr, v), bisect_right(arr, v) - 1


def ans_majority_third_sorted(arr):
    n = len(arr)
    cands = {arr[p] for p in (0, n // 3, 2 * n // 3, n - 1)}
    return sorted(v for v in cands if 3 * ans_count_sorted(arr, v) > n)


assert ans_ex1 == (len(arr_ex1) // 4, next(i for i in range(9) if arr_ex1[i] == arr_ex1[i + 3]))
assert ans_count_sorted([1, 2, 2, 6, 6, 6, 6, 7, 10], 6) == 4
assert ans_count_sorted([1, 2, 2, 6, 6, 6, 6, 7, 10], 5) == 0
assert ans_more_than_1_over_k([1, 1, 2, 2, 2, 3], 3) == [2]
assert ans_more_than_1_over_k([1, 1, 1, 2, 2, 2, 3], 3) == [1, 2]
assert ans_special_block([1, 2, 2, 6, 6, 6, 6, 7, 10]) == (3, 6)
assert ans_majority_third_sorted([1, 1, 1, 2, 3, 3, 3]) == [1, 3]
assert ans_majority_third_sorted([1, 2, 3]) == []
random.seed(29)
for _ in range(3000):
    arr = sorted(random.randint(0, 5) for _ in range(random.randint(1, 30)))
    brute = sorted(v for v, c in Counter(arr).items() if 3 * c > len(arr))
    assert ans_majority_third_sorted(arr) == brute == ans_more_than_1_over_k(arr, 3), arr
print("Đáp án bài tập: tất cả assert PASS")

Đáp án bài tập: tất cả assert PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def findSpecialInteger(self, arr):
        q = len(arr) // 4
        for i in range(len(arr) - q):
            if arr[i] == arr[i + q]:
                return arr[i]
        return -1
```

**Câu thần chú:** *"Sắp xếp ⇒ khối liền nhau; > n/4 ⇒ arr[i] == arr[i + n//4]; muốn log n ⇒ bisect ở n/4, n/2, 3n/4."*

**Checklist:**
- [ ] Điều kiện là **lớn hơn** 25%: `4 * count > n`.
- [ ] Vòng nhảy chỉ tới `len(arr) - q` để không vượt chỉ số.
- [ ] Đã nghĩ tới trường hợp khối ở **đầu** hoặc **cuối** mảng.
- [ ] `n < 4` (ví dụ `[1,1]`, `[5]`) vẫn chạy đúng.